In [ ]:
import pandas as pd
import requests
import os
import subprocess

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

# File paths (raw data stored in data/)
PROTEIN_TARGETS_TSV = f'{DATA_DIR}/05.6_combined_set_protein_targets.tsv'
PROTEIN_TARGETS_XZ  = f'{DATA_DIR}/05.6_combined_set_protein_targets.tsv.xz'
BIOACTIVITIES_TSV   = f'{DATA_DIR}/05.6_combined_set_without_stereochemistry.tsv'
BIOACTIVITIES_XZ    = f'{DATA_DIR}/05.6_combined_set_without_stereochemistry.tsv.xz'
UNIPROT_TSV         = f'{DATA_DIR}/uniprot_human_kinases.tsv'
UNIPROT_GZ          = f'{DATA_DIR}/uniprot_human_kinases.tsv.gz'

PROTEIN_TARGETS_URL = 'https://zenodo.org/records/13817795/files/05.6_combined_set_protein_targets.tsv.xz'
BIOACTIVITIES_URL   = 'https://zenodo.org/records/13817795/files/05.6_combined_set_without_stereochemistry.tsv.xz'
UNIPROT_URL         = 'https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Cid%2Cprotein_name%2Cgene_names%2Clength%2Cmass%2Cxref_chembl%2Cxref_hgnc%2Cgene_primary%2Cec&format=tsv&query=%28%28keyword%3AKW-0418%29+AND+%28reviewed%3Atrue%29%29+AND+%28model_organism%3A9606%29'

# KLIFS kinase UniProt accessions
KINASE_ACCESSIONS = {
    'O00238', 'O00311', 'O00444', 'O00506', 'O14733', 'O14757', 'O14936', 'O14965', 'O14976', 
    'O15075', 'O15264', 'O15530', 'O43293', 'O43318', 'O43353', 'O43781', 'O60674', 'O75116', 
    'O75385', 'O75460', 'O75582', 'O75914', 'O76039', 'O94768', 'O94804', 'O95747', 'O95819', 
    'O96013', 'O96017', 'P00519', 'P00533', 'P04626', 'P04629', 'P06213', 'P06239', 'P06493', 
    'P07332', 'P07333', 'P07949', 'P08069', 'P08581', 'P08631', 'P08922', 'P10721', 'P11309', 
    'P11362', 'P11802', 'P12931', 'P15056', 'P15735', 'P16234', 'P17252', 'P17612', 'P19525', 
    'P19784', 'P21802', 'P21860', 'P22455', 'P22607', 'P23443', 'P23458', 'P24723', 'P24941', 
    'P25098', 'P27037', 'P27361', 'P27448', 'P28482', 'P29317', 'P29320', 'P29322', 'P29597', 
    'P30291', 'P31749', 'P31751', 'P33981', 'P34925', 'P34947', 'P35968', 'P36888', 'P36897', 
    'P37173', 'P41279', 'P41743', 'P42336', 'P42345', 'P42684', 'P43403', 'P43405', 'P45983', 
    'P45984', 'P45985', 'P48729', 'P48730', 'P48736', 'P49137', 'P49336', 'P49759', 'P49760', 
    'P49761', 'P49840', 'P49841', 'P50613', 'P50750', 'P51617', 'P51812', 'P51813', 'P52333', 
    'P52564', 'P53350', 'P53355', 'P53667', 'P53671', 'P53779', 'P54646', 'P54753', 'P54756', 
    'P54760', 'P54762', 'P68400', 'P78362', 'Q00532', 'Q00534', 'Q00535', 'Q00536', 'Q02750', 
    'Q02763', 'Q04759', 'Q04771', 'Q04912', 'Q05397', 'Q05823', 'Q06187', 'Q07912', 'Q08345', 
    'Q08881', 'Q12852', 'Q12866', 'Q13153', 'Q13164', 'Q13188', 'Q13315', 'Q13464', 'Q13523', 
    'Q13535', 'Q13546', 'Q13555', 'Q13557', 'Q13627', 'Q13705', 'Q13882', 'Q13976', 'Q14004', 
    'Q14289', 'Q14680', 'Q15208', 'Q15303', 'Q15375', 'Q15418', 'Q15759', 'Q16288', 'Q16512', 
    'Q16539', 'Q16620', 'Q16644', 'Q16659', 'Q2M2I8', 'Q5TCY1', 'Q6IQ55', 'Q7RTN6', 'Q86Y07', 
    'Q8IU85', 'Q8IV63', 'Q8IVW4', 'Q8IYT8', 'Q8NB16', 'Q8TCG2', 'Q8TF76', 'Q8WU08', 'Q8WZ42', 
    'Q92630', 'Q92772', 'Q92918', 'Q96C45', 'Q96NX5', 'Q96RG2', 'Q96RR4', 'Q96S44', 'Q96SB4', 
    'Q99558', 'Q99640', 'Q99683', 'Q99986', 'Q9BTU6', 'Q9BVS4', 'Q9BYP7', 'Q9H2G2', 'Q9H2K8', 
    'Q9H422', 'Q9H4A3', 'Q9H4B4', 'Q9HAZ1', 'Q9HBH9', 'Q9HCP0', 'Q9NQU5', 'Q9NSY1', 'Q9NWZ3', 
    'Q9NYL2', 'Q9NYV4', 'Q9NYY3', 'Q9NZJ5', 'Q9P286', 'Q9P289', 'Q9UHD2', 'Q9UIK4', 'Q9UKE5', 
    'Q9UM73', 'Q9UQB9', 'Q9UQM7', 'Q9Y2K2', 'Q9Y463', 'Q9Y5S2', 'Q9Y616', 'Q9Y6E0', 'Q9Y6M4'
}

print(f"Data directory: {os.path.abspath(DATA_DIR)}")
print(f"KLIFS accessions loaded: {len(KINASE_ACCESSIONS)}")

Data directory: /home/wilkesbts/Documents/MEP/papyrus_klifs/data
KLIFS accessions loaded: 207


In [43]:
def download_file(url, local_path):
    print(f"Downloading {local_path}...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(local_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    print("Done.")

def decompress(path, cmd='unxz'):
    print(f"Decompressing {path}...")
    subprocess.run([cmd, path], check=True)
    print("Done.")

In [ ]:
# Load protein targets
if not os.path.exists(PROTEIN_TARGETS_TSV):
    download_file(PROTEIN_TARGETS_URL, PROTEIN_TARGETS_XZ)
    decompress(PROTEIN_TARGETS_XZ)

df_protein_targets = pd.read_csv(PROTEIN_TARGETS_TSV, sep='\t', encoding='latin1')
print(f"Protein targets: {df_protein_targets.shape}")
print(df_protein_targets.head())

# Load UniProt human kinases
if not os.path.exists(UNIPROT_TSV):
    download_file(UNIPROT_URL, UNIPROT_GZ)
    decompress(UNIPROT_GZ, cmd='gunzip')

df_uniprot = pd.read_csv(UNIPROT_TSV, sep='\t', encoding='latin1')
print(f"\nUniProt kinases: {df_uniprot.shape}")
print(df_uniprot.head())

Done.
Decompressing data/05.6_combined_set_protein_targets.tsv.xz...
Done.
Protein targets: (7521, 9)
   target_id HGNC_symbol     UniProtID      Status  \
0  P47747_WT         NaN    HRH2_CAVPO    reviewed   
1  B0FL73_WT         NaN  B0FL73_CAVPO  unreviewed   
2  Q8K4Z4_WT         NaN   ADRB2_CAVPO    reviewed   
3  P97266_WT         NaN    OPRM_CAVPO    reviewed   
4  P41144_WT         NaN    OPRK_CAVPO    reviewed   

                       Organism  \
0  Cavia porcellus (Guinea pig)   
1  Cavia porcellus (Guinea pig)   
2  Cavia porcellus (Guinea pig)   
3  Cavia porcellus (Guinea pig)   
4  Cavia porcellus (Guinea pig)   

                                      Classification  Length  \
0  Membrane receptor->Family A G protein-coupled ...   359.0   
1  Membrane receptor->Family A G protein-coupled ...   467.0   
2  Membrane receptor->Family A G protein-coupled ...   418.0   
3  Membrane receptor->Family A G protein-coupled ...    98.0   
4  Membrane receptor->Family A G protein-c

In [ ]:
# Load bioactivities in chunks, filtering to KLIFS accessions and wildtype (WT) proteins
klifs_prefixes = {acc[:6] for acc in KINASE_ACCESSIONS}

if not os.path.exists(BIOACTIVITIES_TSV):
    download_file(BIOACTIVITIES_URL, BIOACTIVITIES_XZ)
    decompress(BIOACTIVITIES_XZ)

chunks = []
for i, chunk in enumerate(pd.read_csv(
    BIOACTIVITIES_TSV, sep='\t', encoding='latin1', chunksize=100_000,
    usecols=['Activity_ID', 'Quality', 'source', 'SMILES', 'target_id',
             'accession', 'Protein_Type', 'pchembl_value', 'Activity_class']
)):
    chunk = chunk[
        chunk['target_id'].str[:6].isin(klifs_prefixes) &
        (chunk['Protein_Type'] == 'WT')
    ]
    chunks.append(chunk)
    if (i + 1) % 20 == 0:
        print(f"  {(i + 1) * 100_000:,} rows processed...")

df_bioactivities = pd.concat(chunks, ignore_index=True)
print(f"Bioactivities loaded (KLIFS-filtered, WT only): {df_bioactivities.shape}")
print(df_bioactivities.head())

Done.
Decompressing data/05.6_combined_set_without_stereochemistry.tsv.xz...
Done.
  2,000,000 rows processed...


/tmp/ipykernel_253906/2852610659.py:10: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(


  4,000,000 rows processed...
  6,000,000 rows processed...
  8,000,000 rows processed...
  10,000,000 rows processed...
  12,000,000 rows processed...
  14,000,000 rows processed...
  16,000,000 rows processed...
  18,000,000 rows processed...
  20,000,000 rows processed...
  22,000,000 rows processed...
  24,000,000 rows processed...
  26,000,000 rows processed...
  28,000,000 rows processed...
  30,000,000 rows processed...
  32,000,000 rows processed...
  34,000,000 rows processed...
  36,000,000 rows processed...
  38,000,000 rows processed...
  40,000,000 rows processed...
  42,000,000 rows processed...
  44,000,000 rows processed...
  46,000,000 rows processed...
  48,000,000 rows processed...
  50,000,000 rows processed...
  52,000,000 rows processed...
  54,000,000 rows processed...
  56,000,000 rows processed...
  58,000,000 rows processed...
Bioactivities loaded (KLIFS-filtered, WT only): (3014541, 9)
                   Activity_ID Quality                          source  \


In [ ]:
# Filter protein_targets to UniProt kinases
uniprot_entry_names = set(df_uniprot['Entry Name'].unique())
df_human_kinase_targets = df_protein_targets[df_protein_targets['UniProtID'].isin(uniprot_entry_names)].copy()
print(f"Targets (UniProt filtered): {df_protein_targets.shape} → {df_human_kinase_targets.shape}")

# Filter KLIFS accessions
df_klifs_targets = df_human_kinase_targets[
    df_human_kinase_targets['target_id'].str[:6].isin(klifs_prefixes)
].copy()
print(f"Targets (KLIFS filtered):   {df_human_kinase_targets.shape} → {df_klifs_targets.shape}")

# Filter to wildtype (WT) only
df_klifs_targets = df_klifs_targets[df_klifs_targets['target_id'].str.endswith('_WT')].copy()
print(f"Targets (WT only):          → {df_klifs_targets.shape}")

# Filter bioactivities
klifs_target_ids = set(df_klifs_targets['target_id'].unique())
df_klifs_interactions = df_bioactivities[df_bioactivities['target_id'].isin(klifs_target_ids)].copy()
print(f"Interactions (KLIFS, WT):   {df_klifs_interactions.shape}")

Targets (UniProt filtered): (7521, 9) → (586, 9)
Targets (KLIFS filtered):   (586, 9) → (258, 9)
Targets (WT only):          → (202, 9)
Interactions (KLIFS, WT):   (3013960, 9)


In [47]:
# Quality analysis: count total entries and non-missing values per quality tier
for qualities in [['Low', 'Medium', 'High'], ['Medium', 'High'], ['High']]:
    df_q = df_klifs_interactions[df_klifs_interactions['Quality'].isin(qualities)]
    print(f"[{'/'.join(qualities)}]  total: {len(df_q):,}")
    print(f"  pchembl_value  (non-null): {df_q['pchembl_value'].count():,}")
    print(f"  Activity_class (non-null): {df_q['Activity_class'].count():,}")
    print()

[Low/Medium/High]  total: 3,013,960
  pchembl_value  (non-null): 379,532
  Activity_class (non-null): 2,634,428

[Medium/High]  total: 270,326
  pchembl_value  (non-null): 270,326
  Activity_class (non-null): 0

[High]  total: 225,125
  pchembl_value  (non-null): 225,125
  Activity_class (non-null): 0



In [48]:
# Save high-quality bioactivities and KLIFS targets to CSV
df_high = df_klifs_interactions[df_klifs_interactions['Quality'] == 'High'].copy()
df_high.to_csv('papyrus_klifs_kinase_bioactivities_high_quality.csv', index=False)
print(f"✓ Saved {len(df_high):,} high-quality bioactivities → papyrus_klifs_kinase_bioactivities_high_quality.csv")

df_klifs_targets.to_csv('papyrus_klifs_kinase_targets.csv', index=False)
print(f"✓ Saved {len(df_klifs_targets):,} KLIFS targets → papyrus_klifs_kinase_targets.csv")

✓ Saved 225,125 high-quality bioactivities → papyrus_klifs_kinase_bioactivities_high_quality.csv
✓ Saved 202 KLIFS targets → papyrus_klifs_kinase_targets.csv
